# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AnanyaGujral/flyrank-ml-internship-start/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

The honest comparison: do learned models beat the transparent rule on **the same held-out clients**?

## 1. Method choice and why

**Features (all knowable at the end of the prior-30 window):** prior-window impressions/clicks/sessions
(log + raw), prior-window CTR, click-per-session, per-client-normalised CTR, prior-window shares of the
client's demand, keyword context, depth, age/freshness, presence flags, and categorical descriptors.
**Nothing** from the outcome window or from the 90-day totals — see `w03_feature_leakage_check.ipynb`.

**Models:**
- **Logistic regression** — the interpretable default; a linear decision boundary in odds space.
- **Random forest** — a non-linear, threshold-friendly comparator; the classic "many small if-then rules voting".
- **Histogram gradient boosting** — the strongest off-the-shelf tabular learner here.

Three models, one split, one metric. If none of them beats the rule, that is the finding.

In [1]:
# ---- shared offline setup: bundled slice, leak-free pool + features, transparent rule ----
import json
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
MIN_PREV_IMPRESSIONS = 100     # "already visible" floor on the TRAILING window
DROP_FACTOR = 0.8              # declined = next-30 impressions < 80% of prior-30

root = Path.cwd().resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
DATA = root / "data" / "raw" / "content_refresh_anonymized.csv"
OUT = root / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
raw = pd.read_csv(DATA)


def build_pool(frame):
    """Already-visible pages with a usable trailing window, plus leak-free features.

    Every feature is measured in the PRIOR 30-day window (days 31-60) or is static
    metadata. Nothing is taken from the 90-day totals or the outcome window, because
    those overlap the label's comparison window.
    """
    pool = frame[frame["impressions_prev_30d"] >= MIN_PREV_IMPRESSIONS].copy()
    pool["declined_next_30d"] = (
        pool["impressions_last_30d"] < DROP_FACTOR * pool["impressions_prev_30d"]).astype(int)
    pool["log_impr_prev"] = np.log1p(pool["impressions_prev_30d"])
    pool["log_clk_prev"] = np.log1p(pool["clicks_prev_30d"])
    pool["log_sess_prev"] = np.log1p(pool["sessions_prev_30d"])
    pool["ctr_prev"] = 100 * pool["clicks_prev_30d"] / pool["impressions_prev_30d"]
    pool["clk_per_sess_prev"] = pool["clicks_prev_30d"] / pool["sessions_prev_30d"].replace(0, np.nan)
    pool["has_keyword"] = pool["search_volume"].notna().astype(int)
    pool["has_clicks_prev"] = (pool["clicks_prev_30d"] > 0).astype(int)
    pool["has_sessions_prev"] = (pool["sessions_prev_30d"] > 0).astype(int)
    g = pool.groupby("client_id")
    pool["ctr_prev_z"] = ((pool["ctr_prev"] - g["ctr_prev"].transform("mean"))
                          / g["ctr_prev"].transform("std").replace(0, np.nan))
    pool["impr_prev_share"] = pool["impressions_prev_30d"] / g["impressions_prev_30d"].transform("sum")
    pool["clk_prev_share"] = pool["clicks_prev_30d"] / g["clicks_prev_30d"].transform("sum").replace(0, np.nan)
    return pool


NUM_FEATURES = ["log_impr_prev", "log_clk_prev", "log_sess_prev", "ctr_prev", "clk_per_sess_prev",
                "ctr_prev_z", "impr_prev_share", "clk_prev_share",
                "search_volume", "competition", "cpc", "word_count", "char_count",
                "content_age_days", "days_since_last_update",
                "has_keyword", "has_clicks_prev", "has_sessions_prev"]
CAT_FEATURES = ["competition_level", "content_type", "main_intent", "age_tier",
                "freshness_tier", "word_count_tier"]


def feature_matrix(pool):
    Xn = pool[NUM_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return pd.concat([Xn, pd.get_dummies(pool[CAT_FEATURES].astype(str), dtype=float)], axis=1)


def rulescore(pool):
    """Transparent rule: low trailing CTR, weighted by trailing exposure."""
    med = float(pool["ctr_prev"].median())
    return (pool["ctr_prev"] <= 0.5 * med).to_numpy() * np.log1p(pool["impressions_prev_30d"].to_numpy())


pool = build_pool(raw)
X = feature_matrix(pool)
y = pool["declined_next_30d"].to_numpy()
groups = pool["client_id"].to_numpy()
rule = rulescore(pool)
print(f"bundled slice: {len(raw):,} rows -> pool {len(pool):,} pages | "
      f"{pool['client_id'].nunique()} pseudo-clients")
print(f"decline base rate in pool: {y.mean():.3f} | feature matrix: {X.shape}")
print("label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)")


def pk(table_scores, y_true, ks=(10, 20, 50, 100)):
    y_true = np.asarray(y_true)
    order = np.argsort(-np.asarray(table_scores))
    return {str(k): round(float(y_true[order[:k]].mean()), 3) for k in ks}

bundled slice: 30,000 rows -> pool 18,010 pages | 30 pseudo-clients
decline base rate in pool: 0.616 | feature matrix: (18010, 43)
label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)


In [2]:
import matplotlib
matplotlib.use("Agg")
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

try:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAVE_HGB = True
except Exception:
    HAVE_HGB = False
print("feature matrix:", X.shape, "| histogram gradient boosting available:", HAVE_HGB)

feature matrix: (18010, 43) | histogram gradient boosting available: True


## 2. Split design

**Grouped by client.** Whole clients are held out (30% of clients, seed 42) so the model is scored on
**new-client scoring** — the deployment case. A random page split would let the model memorise a
client's idiosyncrasies; that inflation is measured in `w06_validation_audit.ipynb`.

In [3]:
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED).split(X, y, groups))
Xtr, Xte, ytr, yte = X.iloc[tr], X.iloc[te], y[tr], y[te]
sc = StandardScaler().fit(Xtr)
print(f"train: {len(tr):,} pages / {pd.Series(groups[tr]).nunique()} clients")
print(f"test : {len(te):,} pages / {pd.Series(groups[te]).nunique()} clients | base rate {yte.mean():.3f}")
print(f"client overlap between train and test: {len(set(groups[tr]) & set(groups[te]))}")

train: 14,439 pages / 21 clients
test : 3,571 pages / 9 clients | base rate 0.567
client overlap between train and test: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as the Week-4 baseline.*

In [4]:
models = {
    "logistic regression": (LogisticRegression(max_iter=5000, random_state=SEED), True),
    "random forest": (RandomForestClassifier(n_estimators=500, min_samples_leaf=10, max_features="sqrt",
                                             random_state=SEED, n_jobs=-1), False),
}
if HAVE_HGB:
    models["hist gradient boosting"] = (HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06,
                                                                       random_state=SEED), False)

scores = {"transparent rule": rule[te]}
for name, (model, scaled) in models.items():
    model.fit(sc.transform(Xtr) if scaled else Xtr, ytr)
    scores[name] = (model.predict_proba(sc.transform(Xte) if scaled else Xte)[:, 1]
                    if name.startswith("logistic") else model.predict_proba(Xte)[:, 1])
scores["rule + best model blend"] = 0.5 * (rule[te] / np.abs(rule[te]).max()) + 0.5 * scores["logistic regression"]

table = {}
print(f"{'method':<26}" + "".join(f" P@{k:<6}" for k in (10, 20, 50, 100)) + "  ROC-AUC")
for name, s in scores.items():
    row = pk(s, yte)
    auc = round(float(roc_auc_score(yte, s)), 3)
    table[name] = {"precision_at_k": row, "roc_auc": auc}
    print(f"{name:<26}" + "".join(f" {row[str(k)]:<8.3f}" for k in (10, 20, 50, 100)) + f"  {auc}")
print(f"\nheld-out base rate: {yte.mean():.3f}  |  test rows: {len(te):,}")

method                     P@10     P@20     P@50     P@100     ROC-AUC
transparent rule           0.700    0.700    0.760    0.670     0.58
logistic regression        0.700    0.750    0.760    0.740     0.585
random forest              0.500    0.500    0.640    0.600     0.582
hist gradient boosting     0.400    0.550    0.580    0.590     0.582
rule + best model blend    0.700    0.750    0.780    0.750     0.61

held-out base rate: 0.567  |  test rows: 3,571


## 4. Errors and interpretation

*Where is the model wrong, and what does it lean on? A short error analysis beats a big metric table.*

In [5]:
from sklearn.inspection import permutation_importance

rf_model = models["random forest"][0]
imp = permutation_importance(rf_model, Xte, yte, n_repeats=5, random_state=SEED, scoring="roc_auc")
order = np.argsort(-imp.importances_mean)[:10]
perm = {X.columns[i]: round(float(imp.importances_mean[i]), 4) for i in order}
print("top permutation importances on HELD-OUT clients (association, not cause):")
for k, v in perm.items():
    print(f"  {k:<28} {v:>8.4f}")

rf_scores = scores["random forest"]
top50 = lambda s: set(np.argsort(-s)[:50])
ov_rule = len(top50(rule[te]) & top50(rf_scores)) 
ov_lr = len(top50(scores["logistic regression"]) & top50(rf_scores))
print(f"\ntop-50 overlap: rule vs RF = {ov_rule} | LR vs RF = {ov_lr} (of 50)")
print("disagreement at the top is the headline error story: the rule and the models are not ranking")
print("the same pages, and on this snapshot the rule is not the weaker of the two.")

top permutation importances on HELD-OUT clients (association, not cause):
  content_age_days               0.0233
  ctr_prev_z                     0.0204
  ctr_prev                       0.0077
  age_tier_365+                  0.0063
  log_sess_prev                  0.0060
  log_clk_prev                   0.0045
  search_volume                  0.0034
  main_intent_transactional      0.0003
  cpc                            0.0002
  main_intent_commercial         0.0001

top-50 overlap: rule vs RF = 1 | LR vs RF = 1 (of 50)
disagreement at the top is the headline error story: the rule and the models are not ranking
the same pages, and on this snapshot the rule is not the weaker of the two.


In [6]:
metrics = {
    "task": "ml-08",
    "data": "data/raw/content_refresh_anonymized.csv",
    "decision_window": "prior-30 (days 31-60)",
    "outcome_window": "later-30 (days 1-30)",
    "split": "grouped-by-client (seed 42, 30% of clients held out)",
    "seed": SEED,
    "test_rows": int(len(te)),
    "test_clients": int(pd.Series(groups[te]).nunique()),
    "test_base_rate": round(float(yte.mean()), 3),
    "table": table,
    "rf_permutation_importance": perm,
    "top50_overlap": {"rule_vs_rf": int(ov_rule), "lr_vs_rf": int(ov_lr)},
}
(OUT / "w05_model_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("receipt written: work/outputs/w05_model_metrics.json")

receipt written: work/outputs/w05_model_metrics.json


### Self-check

- [x] Method choice justified before the numbers
- [x] Split design explained (grouped by client, seed fixed)
- [x] Model and baseline scored on the **same** held-out clients
- [x] Error analysis ties the metrics to real behaviour
- [x] No claim that the model *wins* unless the table says so
- [x] Receipt written to `work/outputs/w05_model_metrics.json`
- [x] Committed under `work/notebooks/`